# A0: Rainfall problem

This notebook implements the rainfall problem in three alternative ways and tests all implementations with the same test cases.

## Rules

- Process rainfall values until the sentinel `-999` is reached.
- Ignore other negative values.
- Zero is a valid rainfall value and is included in the average.
- Return `None` if there are no valid values, so division by zero is handled explicitly.
- Values after `-999` are not processed.

In [ ]:
from itertools import takewhile
from math import isclose

# The sentinel marks the end of the rainfall measurements.
SENTINEL = -999

## Alternative 1: loop with `break` and `continue`

This version keeps a running total and count. It uses an explicit `if` statement to handle the empty-data case.

In [ ]:
def rainfall_loop(readings):
    """Return the average using explicit loop-control statements."""
    total = 0
    count = 0

    for rainfall in readings:
        # Check the sentinel before the general negative-value rule.
        if rainfall == SENTINEL:
            break

        # Negative values other than the sentinel are invalid.
        if rainfall < 0:
            continue

        # Zero is valid and is therefore included in both values.
        total += rainfall
        count += 1

    # No valid values means that an average cannot be calculated.
    if count == 0:
        return None

    return total / count

## Alternative 2: list comprehension

This version finds the sentinel position and uses a list comprehension. It does not use `break` or `continue`, and a conditional expression handles the empty-data case.

In [ ]:
def rainfall_comprehension(readings):
    """Return the average using slicing and a list comprehension."""
    # Work with a new list so that the caller's input is not modified.
    values = list(readings)

    # Ignore every value after the first sentinel.
    end = values.index(SENTINEL) if SENTINEL in values else len(values)

    # Keep zero and positive measurements, but discard invalid negatives.
    valid_values = [rainfall for rainfall in values[:end] if rainfall >= 0]

    # The conditional expression prevents division by zero.
    return sum(valid_values) / len(valid_values) if valid_values else None

## Alternative 3: generator and exception handling

This version uses `takewhile`, a lambda expression, and a generator. It has no `break` or `continue` and handles division by zero with an exception.

In [ ]:
def rainfall_exception(readings):
    """Return the average using lazy iteration and exception handling."""
    # takewhile lazily stops before the sentinel without using break.
    values_before_sentinel = takewhile(
        lambda rainfall: rainfall != SENTINEL,
        readings,
    )

    # A generator filters invalid negatives without creating another list.
    valid_values = (
        rainfall for rainfall in values_before_sentinel if rainfall >= 0
    )

    total = 0
    count = 0

    for rainfall in valid_values:
        total += rainfall
        count += 1

    # In this alternative, the empty-data case is handled by an exception.
    try:
        return total / count
    except ZeroDivisionError:
        return None

## Tests for all implementations

The tests cover normal input, invalid negative values, zero rainfall, values after the sentinel, and both division-by-zero cases. Each function receives a fresh copy of the test data.

In [ ]:
implementations = [
    rainfall_loop,
    rainfall_comprehension,
    rainfall_exception,
]

test_cases = [
    ("normal values", [10, 20, 30, -999], 20.0),
    ("ignore invalid negatives", [-5, 10, -1, 20, -999], 15.0),
    ("zero is valid", [0, 10, -999], 5.0),
    ("stop at sentinel", [10, -999, 100], 10.0),
    ("only invalid values", [-5, -1, -999], None),
    ("sentinel first", [-999], None),
    ("no sentinel", [5, 15], 10.0),
]

# Run exactly the same tests for every implementation.
for implementation in implementations:
    for description, readings, expected in test_cases:
        # Give each function a fresh copy to prevent tests influencing each other.
        actual = implementation(readings.copy())

        if expected is None:
            assert actual is None, (
                f"{implementation.__name__} failed {description}: "
                f"expected None, got {actual}"
            )
        else:
            # isclose is safer than direct equality for floating-point results.
            assert actual is not None and isclose(actual, expected), (
                f"{implementation.__name__} failed {description}: "
                f"expected {expected}, got {actual}"
            )

    print(f"{implementation.__name__}: all tests passed")

print("All rainfall implementations passed all tests.")

## Comparison

- `rainfall_loop` uses explicit control flow and does not create a second list.
- `rainfall_comprehension` creates a filtered list and avoids loop-control statements.
- `rainfall_exception` uses lazy iteration, a lambda expression, and exception handling.